# 5-qubit repetition code: finite-shot SPSA calibration

The 5-data-qubit circuit has 12 independently adjustable CNOT residual
angles. Each gate implements `CX` then target `Rx(theta_g)`. Qiskit Algorithms
SPSA updates selected angles directly; the training loss is $1-P(0^{4})$
with **no injected physical bit flip**. Separately, we measure logical basis
success after syndrome-conditioned recovery on injected-X probes.

Small syndrome loss does not prove that every $\theta_g$ is zero: multiple
gates may have identical or canceling syndrome signatures.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from qec_calibration import (
    SPSAConfig, build_syndrome_template, calibrate_spsa, cnot_locations,
    counts_to_probabilities, logical_success_probability, parameter_norm,
    run_recovery, run_syndrome, single_fault_response_rank,
    syndrome_for_errors, theoretical_single_fault_probabilities,
)

FAST_MODE = False  # True for a quick exploratory run; False for full experiment
MAXITER = 15 if FAST_MODE else 100
TRAIN_SHOTS = 256 if FAST_MODE else 1024
VALIDATION_SHOTS = 1024 if FAST_MODE else 4096
REPEATS = 1 if FAST_MODE else 3
SEED = 42

N = 5
model = build_syndrome_template(N)
locations = pd.DataFrame([{"parameter": i, "gate_id": g.gate_id, "stage": g.stage, "control": g.control, "target": g.target} for i, g in enumerate(model.locations)])
display(locations)
print("Single-fault response rank:", single_fault_response_rank(model), "of", len(model.gate_ids), "angles")


## 1. One-gate sanity check

The objective uses a symbolic Qiskit `ParameterVector` circuit bound by
`SamplerV2` at finite shots. A separate validation run reports pre/post loss.
The callback records every angle after each SPSA iteration.

In [ ]:
single = calibrate_spsa(
    model, {"enc_0_1": 0.6}, active_gate_ids=("enc_0_1",),
    config=SPSAConfig(MAXITER, TRAIN_SHOTS, VALIDATION_SHOTS, seed=SEED),
)
print(f"Initial validation loss: {single.initial_loss:.4f}")
print(f"Final validation loss:   {single.final_loss:.4f}")
print("Final angle:", round(single.optimized_angles["enc_0_1"], 5))
print("SPSA function evaluations:", single.function_evaluations)


## 2. Activate one through 12 CNOT locations

We add faults in a fixed, documented order, using three independent seeded
trials at each size in full mode. The initial values receive small random
perturbations. Finite-shot losses and parameter norms are reported separately;
rank is a diagnostic of isolated syndrome signatures, not a universal
identifiability theorem for nonlinear multi-gate experiments.

In [ ]:
order = ("enc_0_1", "enc_0_2", "enc_0_3", "enc_0_4", "syn_0_left", "syn_1_left", "syn_2_left", "syn_0_right", "syn_1_right", "syn_2_right", "syn_3_left", "syn_3_right")
base = np.asarray([0.6, -0.5, 0.4, -0.35, 0.3, -0.45, 0.25, -0.32, 0.28, -0.38, 0.36, -0.27], dtype=float)
assert len(order) == len(base) == len(model.gate_ids) and set(order) == set(model.gate_ids)
rng = np.random.default_rng(314)
records, results = [], {}
for size in range(1, len(order) + 1):
    active = order[:size]
    rank = single_fault_response_rank(model, active)
    for trial in range(REPEATS):
        start = base[:size] + rng.normal(0, 0.04, size)
        angles = dict(zip(active, start, strict=True))
        config = SPSAConfig(MAXITER, TRAIN_SHOTS, VALIDATION_SHOTS, seed=1000 + 100 * size + trial)
        result = calibrate_spsa(model, angles, active_gate_ids=active, config=config)
        results[(size, trial)] = result
        records.append({
            "active_parameters": size, "trial": trial, "response_rank": rank,
            "initial_loss": result.initial_loss, "final_loss": result.final_loss,
            "initial_norm": parameter_norm(result.initial_angles, active),
            "final_norm": parameter_norm(result.optimized_angles, active),
            "evaluations": result.function_evaluations,
        })
trials = pd.DataFrame(records)
display(trials)
print("Optimization shots across runs:", int((trials.evaluations * TRAIN_SHOTS).sum()))


## 3. Summary across seeded runs

Solid curves are medians; shaded envelopes are the minimum-to-maximum range
of **both** initial and final trials. Three runs are illustrative and do not
define a statistical confidence interval. A zero observed loss at 4,096
shots does not imply an exactly zero underlying error rate.

In [ ]:
summary = trials.groupby("active_parameters", as_index=False).agg(
    rank=("response_rank", "first"),
    initial_loss_med=("initial_loss", "median"), initial_loss_min=("initial_loss", "min"), initial_loss_max=("initial_loss", "max"),
    final_loss_med=("final_loss", "median"), final_loss_min=("final_loss", "min"), final_loss_max=("final_loss", "max"),
    initial_norm_med=("initial_norm", "median"), initial_norm_min=("initial_norm", "min"), initial_norm_max=("initial_norm", "max"),
    final_norm_med=("final_norm", "median"), final_norm_min=("final_norm", "min"), final_norm_max=("final_norm", "max"),
)
display(summary)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), constrained_layout=True)
x = summary.active_parameters.to_numpy()
for ax, metric, ylabel in zip(axes, ("loss", "norm"), (r"Syndrome loss $1-P(0^{n-1})$", r"Angle norm $\|\boldsymbol{\theta}\|_2$")):
    for stage, color in (("initial", "tab:orange"), ("final", "tab:blue")):
        ax.plot(x, summary[f"{stage}_{metric}_med"].to_numpy(), marker="o", color=color, label=f"{stage.title()} median")
        ax.fill_between(x, summary[f"{stage}_{metric}_min"].to_numpy(), summary[f"{stage}_{metric}_max"].to_numpy(), color=color, alpha=0.18, label=f"{stage.title()} trial range")
    ax.set(xlabel="Simultaneously active CNOT angles", ylabel=ylabel, xticks=x)
    ax.set_ylim(bottom=0)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
axes[0].set_title("Circuit-level calibration")
axes[1].set_title("Individual-angle recovery")
plt.show()


## 4. Track all 12 parameters during one representative run

Select the all-gate trial whose final loss is closest to that group's median,
not the best trial. Solid lines are encoding CNOTs; dashed lines are
syndrome-extraction CNOTs. The loss panel is the noisy optimizer callback
loss; the printed final loss above comes from an independent validation.

In [ ]:
all_gate_trials = trials[trials.active_parameters == len(order)].copy()
median_loss = all_gate_trials.final_loss.median()
representative = int((all_gate_trials.final_loss - median_loss).abs().idxmin())
trial_id = int(trials.loc[representative, "trial"])
selected = results[(len(order), trial_id)]
history = pd.DataFrame(selected.history)
fig, axes = plt.subplots(1, 2, figsize=(16, 5.6), constrained_layout=True, gridspec_kw={"width_ratios": [2, 1]})
palette = plt.cm.tab20(np.linspace(0, 1, len(model.gate_ids)))
for idx, gate in enumerate(model.gate_ids):
    angle_index = model.gate_ids.index(gate)
    trajectory = [row[angle_index] for row in history.angles]
    stage = model.locations[angle_index].stage
    axes[0].plot(history.iteration, trajectory, color=palette[idx], linestyle="-" if stage == "encoding" else "--", linewidth=1.6, alpha=0.9, label=gate)
axes[0].axhline(0, color="black", linestyle=":", linewidth=1)
axes[0].set(xlabel="SPSA iteration", ylabel=r"Angle $\theta_g$ (radians)", title="All trainable CNOT parameters")
axes[0].legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
axes[1].plot(history.iteration, history.loss, color="tab:purple")
axes[1].set(xlabel="SPSA iteration", ylabel=r"Sampled $1-P(0^{n-1})$", title="Noisy SPSA objective")
for ax in axes:
    ax.grid(alpha=0.25)
plt.show()
display(pd.DataFrame({"gate_id": model.gate_ids, "initial_angle": [selected.initial_angles[g] for g in model.gate_ids], "final_angle": [selected.optimized_angles[g] for g in model.gate_ids]}))


## 5. Out-of-objective QEC recovery check

The optimizer has seen only the no-error syndrome. We now test its result
with physical X errors and active syndrome-conditioned recovery. Basis-state
logical success is a separate outcome; it does not test phase coherence.

In [ ]:
probes = [("None", ()), *[(f"X{i}", (i,)) for i in range(N)], ("X0X3", (0, 3))]
recovery_rows = []
for i, (label, errors) in enumerate(probes):
    probe = build_syndrome_template(N, physical_errors=errors, recovery=True)
    before = run_recovery(probe, selected.initial_angles, shots=VALIDATION_SHOTS, seed=7000 + i)
    after = run_recovery(probe, selected.optimized_angles, shots=VALIDATION_SHOTS, seed=8000 + i)
    recovery_rows.append({"physical_error": label,
                          "logical_success_before": logical_success_probability(before, 0),
                          "logical_success_after": logical_success_probability(after, 0)})
display(pd.DataFrame(recovery_rows))


## Interpretation

Compare final syndrome loss with final angle norm and the out-of-objective
recovery table. An optimizer can make the chosen syndrome probe look ideal
without uniquely setting each gate parameter to zero. Conversely, short
finite-shot training may leave measurable loss even if a lower-loss solution
exists. The result depends on gate ordering, shot budget, seeds, and the
chosen calibration probes.